# Agent IA USB - entrenamiento (fine-tuning) de Qwen3.5 con Unsloth

Este cuaderno entrena el modelo de Agent IA USB con **conversaciones propias de Agent IA USB** (sus 23 herramientas y respuestas siempre en español) y lo exporta como `.gguf` para la USB.

**Pasos (Google Colab):**
1. Menú **Entorno de ejecución → Cambiar tipo de entorno → GPU T4** → Guardar.
2. Menú **Entorno de ejecución → Ejecutar todo**.
3. Cuando aparezca el botón **Elegir archivos**, selecciona `train.jsonl`, `val.jsonl` y `tools.json` de la carpeta `08_TRAINING\data` de la USB.
4. Espera (≈ 2-3 h para el 4B). Al final el modelo se guarda en tu Google Drive (`MyDrive/agent-ia-usb/`).

**Kaggle:** sube la carpeta `data` como *Dataset*, añádela al notebook, activa **GPU T4** en Settings y pulsa **Run All**. El archivo final queda en *Output*.

> La GPU gratis (T4, 16 GB) alcanza para **Qwen3.5-4B**. El **9B** necesita una GPU de 24 GB o más (Colab Pro L4/A100, o alquilada): cambia `MODEL` abajo.

In [ ]:
# === Ajustes ===
MODEL = "unsloth/Qwen3.5-4B"      # "unsloth/Qwen3.5-9B" solo con GPU de 24 GB+
OUTPUT_NAME = "Qwen3.5-4B-AGENT-IA-USB"    # usa "Qwen3.5-9B-AGENT-IA-USB" si cambias a 9B
MAX_SEQ_LENGTH = 7168              # el prompt de sistema + 23 herramientas de Agent IA USB ocupan ~5k tokens
EPOCHS = 1
LEARNING_RATE = 1e-4
MAX_STEPS = -1                     # pon 30 para una prueba rápida de 10 minutos
SAVE_TO_DRIVE = True               # Colab: guarda el .gguf en Google Drive (más fiable que descargarlo)

In [ ]:
%%capture
# Instala Unsloth (tarda 2-4 minutos)
!pip install --upgrade unsloth

In [ ]:
import os, glob, json

# Busca los datos (Kaggle: /kaggle/input/...; Colab: se suben ahora)
DATA_DIR = None
for base in ["/kaggle/input", "/content/data"]:
    hits = glob.glob(f"{base}/**/train.jsonl", recursive=True)
    if hits:
        DATA_DIR = os.path.dirname(hits[0])
        break
if DATA_DIR is None:
    from google.colab import files
    print("Selecciona train.jsonl, val.jsonl y tools.json (carpeta 08_TRAINING/data de la USB)")
    uploaded = files.upload()
    os.makedirs("/content/data", exist_ok=True)
    for name, content in uploaded.items():
        with open(f"/content/data/{name}", "wb") as f:
            f.write(content)
    DATA_DIR = "/content/data"

TOOLS = json.load(open(f"{DATA_DIR}/tools.json", encoding="utf-8"))
print("Datos en:", DATA_DIR, "| herramientas:", len(TOOLS))

In [ ]:
from unsloth import FastLanguageModel
import torch

# 16-bit LoRA (Unsloth no recomienda QLoRA 4-bit para Qwen3.5)
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=False,
    load_in_16bit=True,
    full_finetuning=False,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=2026,
)
text_tok = getattr(tokenizer, "tokenizer", tokenizer)  # Qwen3.5 may load a multimodal processor
print(torch.cuda.get_device_name(0), f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.0f} GB")

In [ ]:
from datasets import load_dataset

ds = load_dataset("json", data_files={"train": f"{DATA_DIR}/train.jsonl", "val": f"{DATA_DIR}/val.jsonl"})

def to_text(batch):
    texts = []
    for raw in batch["messages_json"]:
        messages = json.loads(raw)
        texts.append(text_tok.apply_chat_template(
            messages, tools=TOOLS, tokenize=False,
            add_generation_prompt=False, enable_thinking=False))
    return {"text": texts}

ds = ds.map(to_text, batched=True, remove_columns=["messages_json"])
sizes = [len(text_tok(t).input_ids) for t in ds["train"]["text"][:40]]
print(f"{len(ds['train'])} train / {len(ds['val'])} val | tokens per example: {min(sizes)}-{max(sizes)}")
if max(sizes) > MAX_SEQ_LENGTH:
    print("WARNING: some examples are longer than MAX_SEQ_LENGTH and will be cut")
print(ds["train"][0]["text"][-1200:])

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=ds["train"],
    eval_dataset=ds["val"].select(range(min(80, len(ds["val"])))),
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LENGTH,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        num_train_epochs=EPOCHS,
        max_steps=MAX_STEPS,
        learning_rate=LEARNING_RATE,
        warmup_ratio=0.05,
        lr_scheduler_type="cosine",
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=100,
        save_strategy="no",
        optim="adamw_8bit",
        weight_decay=0.01,
        seed=2026,
        output_dir="outputs",
        report_to="none",
    ),
)
# Learn only Agent IA USB's own turns (tool calls + replies), not the user/tool text
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)
stats = trainer.train()
print(stats)

In [ ]:
# Quick check: does the trained model call tools and answer in Spanish?
FastLanguageModel.for_inference(model)
system = json.loads(json.loads(open(f"{DATA_DIR}/val.jsonl", encoding="utf-8").readline())["messages_json"])[0]
for q in ["mueve todos los pdf de descargas a D:\\Facturas", "hola, que puedes hacer?", "delete the .tmp files in C:\\Temp"]:
    msgs = [system, {"role": "user", "content": q}]
    prompt = text_tok.apply_chat_template(msgs, tools=TOOLS, tokenize=False,
                                          add_generation_prompt=True, enable_thinking=False)
    inputs = text_tok(prompt, return_tensors="pt").to("cuda")
    out = model.generate(**inputs, max_new_tokens=200, temperature=0.6, top_p=0.8, top_k=20)
    print("USER:", q)
    print(text_tok.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True))
    print("-" * 70)

In [ ]:
# Export to GGUF Q4_K_M (the format Agent IA USB's llama.cpp uses) - takes 10-20 minutes
import shutil
model.save_pretrained_gguf(OUTPUT_NAME, tokenizer, quantization_method="q4_k_m")

candidates = [p for p in glob.glob("**/*.gguf", recursive=True)
              if "mmproj" not in p.lower() and ("q4_k_m" in p.lower() or "Q4_K_M" in p)]
candidates = candidates or [p for p in glob.glob("**/*.gguf", recursive=True) if "mmproj" not in p.lower()]
best = max(candidates, key=os.path.getsize)
final = f"{OUTPUT_NAME}-Q4_K_M.gguf"
shutil.move(best, final)
print("Model ready:", final, f"{os.path.getsize(final) / 1e9:.2f} GB")

In [ ]:
# Save / download the result
if os.path.exists("/kaggle/working"):
    shutil.copy(final, f"/kaggle/working/{final}")
    print("Kaggle: download it from the Output panel on the right:", final)
elif SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs("/content/drive/MyDrive/agent-ia-usb", exist_ok=True)
    shutil.copy(final, f"/content/drive/MyDrive/agent-ia-usb/{final}")
    print("Saved to Google Drive: MyDrive/agent-ia-usb/" + final)
else:
    from google.colab import files
    files.download(final)

## Después: instalarlo en Agent IA USB

1. Copia `Qwen3.5-4B-AGENT-IA-USB-Q4_K_M.gguf` a la carpeta **`08_TRAINING`** de la USB.
2. En la PC, ejecuta: `02_RUNTIME\python\python.exe 08_TRAINING\install_model.py 08_TRAINING\Qwen3.5-4B-AGENT-IA-USB-Q4_K_M.gguf`
   (lo mueve a `01_MODEL`, lo registra y lo compara con el modelo original usando las mismas pruebas).
3. Solo si gana o empata, quedará como modelo por defecto.